# 02 — Merge Human Evaluation with the Sampling Key

This notebook must be executed **after `avaliacao_humana.json` has been completed**.

It:

1. reads `avaliacao_humana.json`;
2. reads `chave_amostragem.json`;
3. merges the files using `avaliacao_id`;
4. validates whether `case_id`, the original case, and the evaluated BDD remain correct;
5. validates scores between 0 and 10;
6. calculates:

\[
\text{Final Score} =
0{,}4 \times \text{Structure}
+
0{,}4 \times \text{Semantics}
+
0{,}2 \times \text{Details}
\]

7. retrieves exactly:
   - model;
   - technique;
   - execution;
   - `generation_id`.

The **`generation_id` field is preserved as the canonical key for the future merge with METEOR, Manhattan, NLI, etc.**  
Therefore, the metric used will always correspond to the **same execution that was evaluated by the human evaluator**.

## Output

`avaliacao_humana_consolidada.json`


In [ ]:
from pathlib import Path
from collections import Counter
import json
import math

## 1. Configuration


In [ ]:
EVALUATION_FILE = Path("avaliacao_humana.json")
SAMPLING_KEY_FILE = Path("chave_amostragem.json")

OUTPUT_FILE = Path("avaliacao_humana_consolidada.json")

MIN_SCORE = 0
MAX_SCORE = 10

# If True, also checks whether the evaluator accidentally changed
# the original text or the sampled BDD.
VALIDATE_TEXTS = True


## 2. Helper Functions


In [ ]:
def load_json(path):
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}"
        )

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2
        )


def normalize_text(text):
    if text is None:
        return None

    # Preserves the content while neutralizing differences caused only
    # by spaces or line breaks.
    return " ".join(str(text).split())


def validate_score(value, criterion, evaluation_id):
    if value is None:
        raise ValueError(
            f"{evaluation_id}: score '{criterion}' was not provided."
        )

    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(
            f"{evaluation_id}: score '{criterion}' must be numeric. "
            f"Value found: {value!r}"
        )

    if not math.isfinite(float(value)):
        raise ValueError(
            f"{evaluation_id}: score '{criterion}' is not finite."
        )

    if not (MIN_SCORE <= float(value) <= MAX_SCORE):
        raise ValueError(
            f"{evaluation_id}: score '{criterion}' is outside the range "
            f"{MIN_SCORE}-{MAX_SCORE}. "
            f"Value found: {value}"
        )

    return float(value)


def calculate_final_score(structure_score, semantics_score, details_score):
    final_score = (
        structure_score * 0.4
        + semantics_score * 0.4
        + details_score * 0.2
    )

    return round(final_score, 2)


## 3. Load Both JSON Files


In [ ]:
human_evaluation = load_json(
    EVALUATION_FILE
)

sampling_key = load_json(
    SAMPLING_KEY_FILE
)

if "avaliacoes" not in human_evaluation:
    raise ValueError(
        "avaliacao_humana.json does not contain the 'avaliacoes' key."
    )

if "selecoes" not in sampling_key:
    raise ValueError(
        "chave_amostragem.json does not contain the 'selecoes' key."
    )

evaluations = human_evaluation["avaliacoes"]
selections = sampling_key["selecoes"]

print(f"Human evaluations: {len(evaluations)}")
print(f"Sampling-key records: {len(selections)}")


## 4. Create Sampling-Key Index by `avaliacao_id`


In [ ]:
key_index = {}

for selection in selections:
    evaluation_id = selection.get("avaliacao_id")

    if not evaluation_id:
        raise ValueError(
            "There is a sampling-key record without avaliacao_id."
        )

    if evaluation_id in key_index:
        raise ValueError(
            f"Duplicate avaliacao_id in the sampling key: {evaluation_id}"
        )

    key_index[evaluation_id] = selection

print(f"Indexed IDs: {len(key_index)}")


## 5. Validate Correspondence Between the Files


In [ ]:
evaluation_ids = []

for evaluation in evaluations:
    evaluation_id = evaluation.get("avaliacao_id")

    if not evaluation_id:
        raise ValueError(
            "There is a human evaluation without avaliacao_id."
        )

    evaluation_ids.append(evaluation_id)

if len(evaluation_ids) != len(set(evaluation_ids)):
    duplicates = [
        item
        for item, count in Counter(evaluation_ids).items()
        if count > 1
    ]

    raise ValueError(
        f"Duplicate avaliacao_id in the human evaluation: {duplicates}"
    )

evaluation_ids = set(evaluation_ids)
key_ids = set(key_index.keys())

missing_in_key = sorted(evaluation_ids - key_ids)
missing_in_evaluation = sorted(key_ids - evaluation_ids)

if missing_in_key:
    raise ValueError(
        f"Evaluations without a matching sampling-key record: {missing_in_key}"
    )

if missing_in_evaluation:
    raise ValueError(
        f"Sampling-key records without a human evaluation: {missing_in_evaluation}"
    )

print("All avaliacao_id values have a 1:1 correspondence.")


## 6. Merge, Validate Scores, and Calculate Final Score


In [ ]:
results = []

for evaluation in evaluations:
    evaluation_id = evaluation["avaliacao_id"]
    key_record = key_index[evaluation_id]

    # --------------------------------------------------------
    # Validate the case
    # --------------------------------------------------------
    evaluation_case_id = evaluation.get("case_id")
    key_case_id = key_record.get("case_id")

    if evaluation_case_id != key_case_id:
        raise ValueError(
            f"{evaluation_id}: divergent case_id. "
            f"Evaluation={evaluation_case_id}, key={key_case_id}"
        )

    # --------------------------------------------------------
    # Validate texts, if enabled
    # --------------------------------------------------------
    if VALIDATE_TEXTS:
        evaluation_original = normalize_text(
            evaluation.get("caso_original")
        )

        key_original = normalize_text(
            key_record.get("original_case")
        )

        if evaluation_original != key_original:
            raise ValueError(
                f"{evaluation_id}: the original case was changed "
                f"between the sampling key and the evaluation."
            )

        evaluation_bdd = normalize_text(
            evaluation.get("bdd_gerado")
        )

        key_bdd = normalize_text(
            key_record.get("gherkin")
        )

        if evaluation_bdd != key_bdd:
            raise ValueError(
                f"{evaluation_id}: the evaluated BDD does not match "
                f"the BDD sampled in the key."
            )

    # --------------------------------------------------------
    # Read and validate scores
    # --------------------------------------------------------
    scores = evaluation.get("avaliacao", {})

    structure_score = validate_score(
        scores.get("estrutura"),
        "estrutura",
        evaluation_id
    )

    semantics_score = validate_score(
        scores.get("semantica"),
        "semantica",
        evaluation_id
    )

    details_score = validate_score(
        scores.get("detalhes"),
        "detalhes",
        evaluation_id
    )

    final_score = calculate_final_score(
        structure_score,
        semantics_score,
        details_score
    )

    # --------------------------------------------------------
    # Consolidated record
    #
    # generation_id is the key that must be used later
    # to retrieve the metrics from the exact same execution.
    # --------------------------------------------------------
    record = {
        # Compatibility contract: persisted field names remain unchanged.
        "avaliacao_id": evaluation_id,
        "case_id": key_record["case_id"],
        "source_id": key_record.get("source_id"),
        "source_line": key_record.get("source_line"),

        "caso_original": key_record.get("original_case"),
        "bdd_gerado": key_record.get("gherkin"),

        "model": key_record.get("model"),
        "technique": key_record.get("technique"),
        "execution": int(key_record.get("execution")),
        "generation_id": key_record.get("generation_id"),

        "estrutura": structure_score,
        "semantica": semantics_score,
        "detalhes": details_score,
        "nota_final": final_score
    }

    results.append(record)

print(f"Consolidated records: {len(results)}")


## 7. Validate `generation_id` and the Exact Execution


In [ ]:
generation_ids = [
    item["generation_id"]
    for item in results
]

if any(not generation_id for generation_id in generation_ids):
    raise ValueError(
        "There is a consolidated record without generation_id."
    )

if len(generation_ids) != len(set(generation_ids)):
    duplicates = [
        item
        for item, count in Counter(generation_ids).items()
        if count > 1
    ]

    raise ValueError(
        f"Duplicate generation_id: {duplicates}"
    )

# Additional validation:
# the execution suffix must be compatible with the generation_id.
for item in results:
    expected_suffix = f"__exec-{item['execution']:02d}"

    if not item["generation_id"].endswith(expected_suffix):
        raise ValueError(
            f"{item['avaliacao_id']}: generation_id does not match "
            f"execution {item['execution']}. "
            f"generation_id={item['generation_id']}"
        )

print(
    "All generation_id values are unique and match "
    "the recorded execution."
)


## 8. Generate the Consolidated JSON


In [ ]:
output_data = {
    "metadata": {
        "total_avaliacoes": len(results),
        "escala": {
            "minimo": MIN_SCORE,
            "maximo": MAX_SCORE
        },
        "pesos": {
            "estrutura": 0.4,
            "semantica": 0.4,
            "detalhes": 0.2
        },
        "formula_nota_final": (
            "(estrutura * 0.4) + "
            "(semantica * 0.4) + "
            "(detalhes * 0.2)"
        ),
        "chave_para_cruzamento_com_metricas": "generation_id",
        "observacao_metricas": (
            "O generation_id identifica exatamente o caso, modelo, "
            "tecnica e execucao avaliados pelo humano."
        ),
        "arquivos_origem": {
            "avaliacao_humana": str(EVALUATION_FILE),
            "chave_amostragem": str(SAMPLING_KEY_FILE)
        }
    },
    "resultados": results
}

save_json(
    OUTPUT_FILE,
    output_data
)

print(f"Generated: {OUTPUT_FILE.resolve()}")


## 9. Summary of Scores and Experimental Conditions


In [ ]:
final_scores = [
    item["nota_final"]
    for item in results
]

mean_score = sum(final_scores) / len(final_scores)
minimum_score = min(final_scores)
maximum_score = max(final_scores)

print("=" * 70)
print("CONSOLIDATION COMPLETED")
print("=" * 70)

print(f"Total: {len(results)}")
print(f"Mean final score: {mean_score:.2f}")
print(f"Lowest score: {minimum_score:.2f}")
print(f"Highest score: {maximum_score:.2f}")

print("\nBy model:")
for model, count in sorted(
    Counter(item["model"] for item in results).items()
):
    print(f"  {model}: {count}")

print("\nBy technique:")
for technique, count in sorted(
    Counter(item["technique"] for item in results).items()
):
    print(f"  {technique}: {count}")

print("\nBy execution:")
for execution in range(1, 11):
    count = sum(
        1
        for item in results
        if item["execution"] == execution
    )
    print(f"  Execution {execution}: {count}")

print(f"\nFinal file: {OUTPUT_FILE}")


## 10. Next Step: Automatic Metrics

The `avaliacao_humana_consolidada.json` file is now ready for the correlation analysis.

The recommended key for merging with metric files is:

```text
generation_id
```

Example:

```text
TC_177__google-gemma-4-e4b-it__one-shot__exec-07
```

This key prevents the human score for execution 7 from being incorrectly associated with metrics from executions 1, 2, 3, etc.

As a redundant validation, in the future metric-processing step it is also recommended to confirm:

- `case_id`
- `model` / `modelo`
- `technique` / `tecnica`
- `execution` / `execucao`
